In [ ]:
import gensim
from nltk.cluster.kmeans import KMeansClusterer
from nltk.cluster.util import cosine_distance, euclidean_distance
from os import path
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import numpy as np
from gensim.summarization import keywords
from gensim.summarization import summarize
from summarization import Summarizer
from language_tool import LanguageTool
import pickle

In [ ]:
#SKIP
model = gensim.models.KeyedVectors.load_word2vec_format('./GoogleNews-vectors-negative300.bin', binary=True)
#https://drive.google.com/uc?id=0B7XkCwpI5KDYNlNUTTlSS21pQmM&export=download

In [ ]:
#SKIP
folder = "samples3"
summarizer = Summarizer(folder)
sums = summarizer.summarize_all()
files = summarizer.files
sources = summarizer.sources

for sum in sums:
    print(len(sum))

corpus = []
d = dict()

i = 0
for sum in sums:
    for x in sum:
        corpus.append(x)
        d[x] = i
        i+=1
        
print("\n",len(corpus))
pickle.dump(sources, open("sources.p", "wb" ))
pickle.dump(corpus, open("corpus.p", "wb" ))
pickle.dump(files, open("files.p", "wb" ))
pickle.dump(sums, open("sums.p", "wb" ))
pickle.dump(d, open("d.p", "wb" ))

In [ ]:
corpus = pickle.load(open("corpus.p", "rb" ) )
model_small = pickle.load(open("model.p", "rb" ) )
data = []
for sentence in corpus:
    sentence = sentence.lower().split()
    #res = []
    res = 0
    n = 0
    for word in sentence:
        try:
            #model_small[word]=model[word]
            res+=model_small[word]
            #res.append(model[word])
            n+=1
        except:
            pass
    res/=n
    #print(n)
    #for i in range(50-n):
    #    res.append([0]*300)
    res = np.array(res)
    #print(res.shape)
    data.append(res.reshape(-1))
    
data = np.array(data)
pickle.dump(data, open( "data.p", "wb" ) )
#pickle.dump(model_small, open( "model.p", "wb" ) )

In [ ]:
data = pickle.load(open("data.p", "rb" ) )
NUM_CLUSTERS = 3
import random

kclusterer = KMeansClusterer(NUM_CLUSTERS, distance=euclidean_distance, repeats=100, rng=random.Random(0,5)) #euclidean_distance #cosine_distance
assigned_clusters = kclusterer.cluster(data, assign_clusters=True)
#print(kclusterer.means())

import re
from collections import Counter
for j in range(NUM_CLUSTERS):
    cluster = [corpus[i] for i in range(len(corpus)) if assigned_clusters[i]==j]

    text = []
    for i in range(len(cluster)):
        sentense = str(cluster[i]).lower().split(" ")
        for word in sentense:
            text.append(re.sub(r'[^\w\s]','',str(word)))
    cnt = Counter(text)
    f = open("cluster_"+str(j)+".txt", 'w')
    for item in text:
        f.write("%s\n" % item)
    f.close()
    
    text = open("cluster_"+str(j)+".txt").read()
    wordcloud = WordCloud().generate(text)
    
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    m = 100
    l = 0
    for k in range(len(data)):
        t = euclidean_distance(data[k], kclusterer.means()[j])
        if t<m:
            m=t
            l=k
    print(corpus[l])
    plt.show()

In [ ]:
weights = [0.59, 0.11, 0.37]

In [ ]:
sources = pickle.load(open("sources.p", "rb" ) )
files = pickle.load(open("files.p", "rb" ) )
sums = pickle.load(open("sums.p", "rb" ) )
d = pickle.load(open("d.p", "rb" ) )
i = 0
score = dict()
for essay in sums:
    res = 0
    for sentance in essay:
        k = assigned_clusters[d[sentance]]
        #print(k)
        res+= weights[k]
    print(files[i], round(res,2))
    score[files[i]] = res
    i+=1

In [ ]:
import nltk, string
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import matplotlib.pyplot as plt
import operator

#nltk.download('punkt') # if necessary...

stemmer = nltk.stem.porter.PorterStemmer()
remove_punctuation_map = dict((ord(char), None) for char in string.punctuation)

def stem_tokens(tokens):
    return [stemmer.stem(item) for item in tokens]

'''remove punctuation, lowercase, stem'''
def normalize(text):
    return stem_tokens(nltk.word_tokenize(text.lower().translate(remove_punctuation_map)))

vectorizer = TfidfVectorizer(tokenizer=normalize, stop_words='english')

def cosine_sim(text1, text2):
    tfidf = vectorizer.fit_transform([text1, text2])
    return ((tfidf * tfidf.T).A)[0,1]

origin = dict()
for i in range(len(sources)):
    origin[files[i]] = 0
    for j in range(len(sources)):
        origin[files[i]] += 1/(1.01 - cosine_sim(sources[i], sources[j]))
    origin[files[i]] =  1/origin[files[i]]*100 #len(sources)/2   
    #print(files[i], origin[files[i]])

similatity = dict()
for i in range(len(sources)):
    for j in range(i+1,len(sources)):
        similatity[files[i] + " " + files[j]] = cosine_sim(sources[i], sources[j])
        #print(files[i], files[j], cosine_sim(sources[i], sources[j]))

cnt = Counter(similatity)

sorted_cnt = sorted(cnt.items(), key=operator.itemgetter(1), reverse=True)
#print(sorted_cnt)

labels, values = zip(*sorted_cnt)

indexes = np.arange(len(labels))
width = 1

plt.bar(indexes, values, width)
plt.xticks(indexes + width * 0.5, [])
plt.title('Sorted similarity by TF-IDF')
plt.show()

print("Top 10:")
cnt.most_common(10)

In [ ]:
#Yule?s K characteristic
from nltk.stem.porter import PorterStemmer
from itertools import groupby
 
def words(entry):
    return filter(lambda w: len(w) > 0,
                  [w.strip("0123456789!:,.?(){}[]") for w in entry.split()])
 
def yule(entry):
    # yule's I measure (the inverse of yule's K measure)
    # higher number is higher diversity - richer vocabulary
    d = dict()
    stemmer = PorterStemmer()
    for w in words(entry):
        w = stemmer.stem(w).lower()
        try:
            d[w] += 1
        except KeyError:
            d[w] = 1
    M1 = float(len(d))
    M2 = np.sum([len(list(g))*(freq**2) for freq,g in groupby(sorted(d.values()))])
    #print(M1,M2)
    try:
        return (M1*M1)/(M2-M1)
    except ZeroDivisionError:
        return 0

diversity = dict()
for i in range(len(sources)):
    diversity[files[i]] = yule(sources[i])/20
    #print(files[i], yule(sources[i]))
    
cnt = Counter(diversity)

sorted_cnt = sorted(cnt.items(), key=operator.itemgetter(1), reverse=True)
#print(sorted_cnt)

labels, values = zip(*sorted_cnt)

indexes = np.arange(len(labels))
width = 1

plt.bar(indexes, values, width)
plt.xticks(indexes + width * 0.5, [])
plt.title('Yule\'s K measure of vocabulary richness')
plt.show()

print("Top 10:")
cnt.most_common(10)

In [ ]:
penalty = LanguageTool().get_check_summary_from_list(sources)
penalty

In [ ]:
grammar = []
spelling = []
style = []
for x in penalty:
    grammar.append(-x["grammar"]/10)
    spelling.append(-x["spelling"]/10)
    style.append(-x["style"]/10)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
min_max_scaler = MinMaxScaler()

result = pd.DataFrame({"NAME":files})

result["Content"] = score.values() #min_max_scaler.fit_transform(list(score.values()))
result["Vocabulary"] = diversity.values() #min_max_scaler.fit_transform(list(diversity.values()))
result["Authenticity"] = origin.values() #min_max_scaler.fit_transform(list(origin.values()))
result["Grammar"] = grammar
result["Spelling"] = spelling
result["Style"] = style
result["SCORE"] = result["Content"]+result["Vocabulary"]+result["Authenticity"]

result.round(2)